In [ ]:
# 1. Imports and setup

from pathlib import Path 
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Config and paths

PROJECT_ROOT = Path.cwd().parent

# Detect whether we are running on Kaggle or locally
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_ROOT = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")
    OUTPUT_DIR = Path("/kaggle/working")
    CHECKPOINT_PATH = OUTPUT_DIR / "best_model.pt"
else:
    DATA_ROOT = PROJECT_ROOT / "recodai-luc-scientific-image-forgery-detection"
    OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "notebook_outputs"
    CHECKPOINT_PATH = PROJECT_ROOT / "artifacts" / "checkpoints" / "best_model.pt"

# Training / inference settings
IMAGE_SIZE = (512, 512)
BATCH_SIZE = 4
THRESHOLD = 0.5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("IS_KAGGLE:", IS_KAGGLE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("IMAGE_SIZE:", IMAGE_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("THRESHOLD:", THRESHOLD)

In [ ]:
# 3. U-Net model definition

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(kernel_size=2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.block(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            in_channels,
            in_channels // 2,
            kernel_size=2,
            stride=2
        )
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        diff_y = x2.size()[2] - x1.size()[2]
        diff_x = x2.size()[3] - x1.size()[3]

        x1 = nn.functional.pad(
            x1,
            [diff_x // 2, diff_x - diff_x // 2,
             diff_y // 2, diff_y - diff_y // 2]
        )

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)

        self.outc = OutConv(64, out_channels)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        logits = self.outc(x)
        return logits

In [ ]:
# 4. Create model

model = UNet(in_channels=3, out_channels=1).to(device)

print("Model created successfully.")
print("Model device:", next(model.parameters()).device)

In [ ]:
# 5. Load dataset metadata

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


def _extract_case_id(path):
    return path.stem


def _list_image_files(folder):
    return sorted(
        [
            path
            for path in folder.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        ]
    )


def _list_npy_files(folder):
    return sorted(
        [
            path
            for path in folder.iterdir()
            if path.is_file() and path.suffix.lower() == ".npy"
        ]
    )


def _build_mask_lookup(mask_folder):
    mask_files = _list_npy_files(mask_folder)
    mask_lookup = {}

    for mask_path in mask_files:
        case_id = _extract_case_id(mask_path)
        mask_lookup[case_id] = mask_path

    return mask_lookup


def _build_image_rows(image_folder, image_type, split, mask_lookup=None):
    image_files = _list_image_files(image_folder)
    rows = []

    for image_path in image_files:
        case_id = _extract_case_id(image_path)

        has_mask = False
        num_masks = 0
        mask_paths = []

        if image_type == "forged" and mask_lookup is not None:
            matched_mask = mask_lookup.get(case_id)

            if matched_mask is not None:
                has_mask = True
                num_masks = 1
                mask_paths = [matched_mask]

        rows.append(
            {
                "case_id": case_id,
                "image_path": image_path,
                "mask_paths": mask_paths,
                "num_masks": num_masks,
                "split": split,
                "image_type": image_type,
                "has_mask": has_mask,
            }
        )

    return rows


def load_dataset_metadata(dataset_root):
    dataset_root = Path(dataset_root)

    train_authentic_folder = dataset_root / "train_images" / "authentic"
    train_forged_folder = dataset_root / "train_images" / "forged"
    train_masks_folder = dataset_root / "train_masks"

    supplemental_folder = dataset_root / "supplemental_images"
    supplemental_masks_folder = dataset_root / "supplemental_masks"

    train_mask_lookup = _build_mask_lookup(train_masks_folder)

    rows = []

    rows.extend(
        _build_image_rows(
            image_folder=train_authentic_folder,
            image_type="authentic",
            split="train",
            mask_lookup=None,
        )
    )

    rows.extend(
        _build_image_rows(
            image_folder=train_forged_folder,
            image_type="forged",
            split="train",
            mask_lookup=train_mask_lookup,
        )
    )

    if supplemental_folder.exists():
        supplemental_mask_lookup = _build_mask_lookup(supplemental_masks_folder)

        rows.extend(
            _build_image_rows(
                image_folder=supplemental_folder,
                image_type="forged",
                split="supplemental",
                mask_lookup=supplemental_mask_lookup,
            )
        )

    df = pd.DataFrame(rows)

    df["_case_id_sort"] = pd.to_numeric(df["case_id"])
    df = df.sort_values(["_case_id_sort", "case_id"]).drop(columns="_case_id_sort")
    df = df.reset_index(drop=True)

    return df


df = load_dataset_metadata(DATA_ROOT)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nImage type counts:")
print(df["image_type"].value_counts())

print("\nSplit counts:")
print(df["split"].value_counts())

print("\nHas-mask counts:")
print(df["has_mask"].value_counts())

In [ ]:
# 6. PyTorch dataset class

class BiomedicalForgeryDataset(Dataset):
    def __init__(self, df, target_size=(512, 512), transforms=None):
        self.df = df.reset_index(drop=True)
        self.target_size = target_size
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def _resize_image_and_mask(self, image, mask):
        target_h, target_w = self.target_size

        image = cv2.resize(
            image,
            (target_w, target_h),
            interpolation=cv2.INTER_LINEAR
        )

        mask = cv2.resize(
            mask,
            (target_w, target_h),
            interpolation=cv2.INTER_NEAREST
        )

        return image, mask

    def _load_image(self, row):
        image_path = Path(row["image_path"])
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

        if image is None:
            raise ValueError(f"Could not read image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

    def _load_instance_masks(self, row, height, width):
        if not row["has_mask"]:
            return []

        mask_path = Path(row["mask_paths"][0])
        mask = np.load(mask_path)

        if mask.ndim == 2:
            if mask.shape != (height, width):
                raise ValueError(
                    f"Mask shape {mask.shape} does not match image shape {(height, width)} "
                    f"for case_id={row['case_id']}"
                )
            return [(mask > 0).astype(np.uint8)]

        if mask.ndim == 3:
            if mask.shape[1:] != (height, width):
                raise ValueError(
                    f"Mask spatial shape {mask.shape[1:]} does not match image shape {(height, width)} "
                    f"for case_id={row['case_id']}"
                )

            instance_masks = []
            for i in range(mask.shape[0]):
                instance_masks.append((mask[i] > 0).astype(np.uint8))

            return instance_masks

        raise ValueError(
            f"Unsupported mask shape {mask.shape} for case_id={row['case_id']}"
        )

    def _merge_instance_masks(self, instance_masks, height, width):
        if len(instance_masks) == 0:
            return np.zeros((height, width), dtype=np.uint8)

        merged_mask = np.zeros((height, width), dtype=np.uint8)

        for instance_mask in instance_masks:
            merged_mask = np.logical_or(merged_mask, instance_mask > 0)

        return merged_mask.astype(np.uint8)

    def get_sample_info(self, idx):
        row = self.df.iloc[idx]

        image = self._load_image(row)
        height, width = image.shape[:2]

        instance_masks = self._load_instance_masks(row, height, width)

        return {
            "case_id": row["case_id"],
            "image_type": row["image_type"],
            "has_mask": row["has_mask"],
            "original_size": (height, width),
            "num_instances": len(instance_masks),
            "instance_masks": instance_masks,
        }

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = self._load_image(row)
        height, width = image.shape[:2]

        instance_masks = self._load_instance_masks(row, height, width)
        mask = self._merge_instance_masks(instance_masks, height, width)

        image, mask = self._resize_image_and_mask(image, mask)

        mask = (mask > 0).astype(np.uint8)

        if self.transforms is not None:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image).float()
            image = image.permute(2, 0, 1) / 255.0

        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask).float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        mask = (mask > 0).float()

        return image, mask

In [ ]:
# 7.  Loading Dataset and sanity check

dataset = BiomedicalForgeryDataset(
    df=df,
    target_size=IMAGE_SIZE,
    transforms=None
)

print("Number of samples in dataset:", len(dataset))

# Prefer a forged sample with a mask for visual checking
forged_indices = df.index[df["has_mask"] == True].tolist()

if len(forged_indices) > 0:
    sample_idx = forged_indices[0]
else:
    sample_idx = 0

image, mask = dataset[sample_idx]
info = dataset.get_sample_info(sample_idx)

print("\n--- Single sample check ---")
print("Sample index:", sample_idx)
print("Case ID:", info["case_id"])
print("Image type:", info["image_type"])
print("Has mask:", info["has_mask"])
print("Original size:", info["original_size"])
print("Number of instance masks:", info["num_instances"])

print("\nReturned tensor shapes:")
print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("\nReturned tensor dtypes:")
print("Image dtype:", image.dtype)
print("Mask dtype:", mask.dtype)

print("\nReturned tensor value ranges:")
print("Image min:", image.min().item())
print("Image max:", image.max().item())
print("Mask unique values:", torch.unique(mask))
print("Mask pixel sum:", mask.sum().item())

image_np = image.permute(1, 2, 0).cpu().numpy()
mask_np = mask.squeeze(0).cpu().numpy()
mask_np = (mask_np > 0).astype(float)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_np)
axes[0].set_title("Processed Image")
axes[0].axis("off")

axes[1].imshow(mask_np, cmap="gray")
axes[1].set_title("Processed Mask")
axes[1].axis("off")

axes[2].imshow(image_np)
axes[2].imshow(mask_np, cmap="Reds", alpha=0.4)
axes[2].set_title("Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 8. Training setup

# Train/validation split
VAL_FRACTION = 0.2

num_samples = len(dataset)
num_val = int(num_samples * VAL_FRACTION)
num_train = num_samples - num_val

train_dataset, val_dataset = random_split(
    dataset,
    [num_train, num_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


# Loss function
bce_loss = nn.BCEWithLogitsLoss()


def dice_loss(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)

    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (probs * targets).sum(dim=1)
    union = probs.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + smooth) / (union + smooth)

    return 1.0 - dice.mean()


def combined_loss(logits, targets):
    return bce_loss(logits, targets) + dice_loss(logits, targets)


# Validation metric
def dice_score_from_logits(logits, targets, threshold=0.5, smooth=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + smooth) / (union + smooth)

    return dice.mean().item()


# Optimizer
LEARNING_RATE = 1e-4

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Training setup complete.")
print("Learning rate:", LEARNING_RATE)

In [ ]:
# 9. Training and validation functions

def train_one_epoch(model, loader, optimizer, loss_fn, device, max_batches=None):
    model.train()

    running_loss = 0.0
    num_seen = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch_idx, (images, masks) in enumerate(progress_bar):
        if max_batches is not None and batch_idx >= max_batches:
            break

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = loss_fn(logits, masks)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)

        running_loss += loss.item() * batch_size
        num_seen += batch_size

        progress_bar.set_postfix(loss=loss.item())

    epoch_loss = running_loss / max(num_seen, 1)

    return epoch_loss


def validate_one_epoch(model, loader, loss_fn, device, threshold=0.5, max_batches=None):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    num_seen = 0

    progress_bar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(progress_bar):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = loss_fn(logits, masks)

            dice = dice_score_from_logits(
                logits=logits,
                targets=masks,
                threshold=threshold
            )

            batch_size = images.size(0)

            running_loss += loss.item() * batch_size
            running_dice += dice * batch_size
            num_seen += batch_size

            progress_bar.set_postfix(
                loss=loss.item(),
                dice=dice
            )

    epoch_loss = running_loss / max(num_seen, 1)
    epoch_dice = running_dice / max(num_seen, 1)

    return epoch_loss, epoch_dice

In [ ]:
# 10. Run training with runtime limit and keep best model in memory

EPOCHS = 4

# Runtime control
MAX_RUNTIME_HOURS = 3.9
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 60 * 60

# Optional batch limits
# Use None for full training.
# Use numbers like 300 or 500 if full epochs are too slow.
MAX_TRAIN_BATCHES = 75
MAX_VAL_BATCHES = 75

# Early stopping
PATIENCE = 2

best_val_score = -1.0
best_model_state = None
history = []

epochs_without_improvement = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    elapsed = time.time() - start_time

    if elapsed >= MAX_RUNTIME_SECONDS:
        print("\nStopping training because runtime limit was reached.")
        break

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"Elapsed time: {elapsed / 60:.1f} minutes")

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        loss_fn=combined_loss,
        device=device,
        max_batches=MAX_TRAIN_BATCHES
    )

    elapsed = time.time() - start_time

    if elapsed >= MAX_RUNTIME_SECONDS:
        print("\nStopping before validation because runtime limit was reached.")
        break

    val_loss, val_dice = validate_one_epoch(
        model=model,
        loader=val_loader,
        loss_fn=combined_loss,
        device=device,
        threshold=THRESHOLD,
        max_batches=MAX_VAL_BATCHES
    )

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_dice": val_dice,
            "elapsed_minutes": (time.time() - start_time) / 60,
        }
    )

    print(
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    if val_dice > best_val_score:
        best_val_score = val_dice
        epochs_without_improvement = 0

        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        print("Best model updated in memory.")
        print(f"Best validation Dice: {best_val_score:.4f}")

    else:
        epochs_without_improvement += 1

        print(
            f"No improvement for {epochs_without_improvement} epoch(s)."
        )

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping triggered.")
            break


history_df = pd.DataFrame(history)

print("\nTraining complete.")
print("Best validation Dice:", best_val_score)
print(f"Total elapsed time: {(time.time() - start_time) / 60:.1f} minutes")

display(history_df)

In [ ]:
# 11. Plot training history

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_df["epoch"], history_df["val_dice"], marker="o", label="Validation Dice")
axes[1].set_title("Validation Dice Score")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice Score")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 12. Load best trained checkpoint for inference

def load_checkpoint(path, model, map_location="cpu"):
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint


checkpoint = load_checkpoint(
    path=CHECKPOINT_PATH,
    model=model,
    map_location=device
)

print("Checkpoint loaded successfully.")

if "epoch" in checkpoint:
    print("Saved epoch:", checkpoint["epoch"])

if "best_val_score" in checkpoint:
    print("Best validation score:", checkpoint["best_val_score"])

if "image_size" in checkpoint:
    print("Checkpoint image size:", checkpoint["image_size"])

if "threshold" in checkpoint:
    print("Checkpoint threshold:", checkpoint["threshold"])

model.eval()
print("Model set to evaluation mode.")

In [ ]:
# 13. Build inference-only metadata table

def build_inference_metadata(image_folder):
    image_folder = Path(image_folder)

    if not image_folder.exists():
        raise FileNotFoundError(f"Inference image folder not found: {image_folder}")

    image_files = sorted(
        [
            path
            for path in image_folder.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        ]
    )

    rows = []

    for image_path in image_files:
        case_id = image_path.stem

        rows.append(
            {
                "case_id": case_id,
                "image_path": image_path,
            }
        )

    inference_df = pd.DataFrame(rows)

    inference_df["_case_id_sort"] = pd.to_numeric(inference_df["case_id"], errors="coerce")
    inference_df = inference_df.sort_values(["_case_id_sort", "case_id"])
    inference_df = inference_df.drop(columns="_case_id_sort")
    inference_df = inference_df.reset_index(drop=True)

    return inference_df


# Use test images for Kaggle submission.
# If test_images is not available locally, fall back to train forged images for debugging.
TEST_IMAGE_FOLDER = DATA_ROOT / "test_images"
DEBUG_IMAGE_FOLDER = DATA_ROOT / "train_images" / "forged"

if TEST_IMAGE_FOLDER.exists():
    INFERENCE_IMAGE_FOLDER = TEST_IMAGE_FOLDER
    print("Using test images for inference.")
else:
    INFERENCE_IMAGE_FOLDER = DEBUG_IMAGE_FOLDER
    print("test_images folder not found. Using train forged images for local debugging.")

inference_df = build_inference_metadata(INFERENCE_IMAGE_FOLDER)

print("Inference image folder:", INFERENCE_IMAGE_FOLDER)
print("Inference dataframe shape:", inference_df.shape)

print("\nColumns:")
print(inference_df.columns.tolist())

print("\nFirst 5 rows:")
display(inference_df.head())

In [ ]:
# 14. Inference-only dataset class

class InferenceForgeryDataset(Dataset):
    def __init__(self, df, target_size=(512, 512)):
        self.df = df.reset_index(drop=True)
        self.target_size = target_size

    def __len__(self):
        return len(self.df)

    def _load_image(self, image_path):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

        if image is None:
            raise ValueError(f"Could not read image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

    def _resize_image(self, image):
        target_h, target_w = self.target_size
        image = cv2.resize(
            image,
            (target_w, target_h),
            interpolation=cv2.INTER_LINEAR
        )
        return image

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        case_id = row["case_id"]
        image_path = Path(row["image_path"])

        image = self._load_image(image_path)
        original_height, original_width = image.shape[:2]

        image = self._resize_image(image)

        image = torch.from_numpy(image).float()
        image = image.permute(2, 0, 1) / 255.0

        return {
            "case_id": case_id,
            "image": image,
            "original_size": (original_height, original_width),
        }

In [ ]:
# 15. RLE encoding helper

def mask_to_rle(mask):
    
    mask = (mask > 0).astype(np.uint8)

    flat_mask = mask.flatten(order="F")

    padded = np.concatenate([[0], flat_mask, [0]])
    changes = np.where(padded[1:] != padded[:-1])[0] + 1

    starts = changes[::2]
    ends = changes[1::2]
    lengths = ends - starts

    if len(starts) == 0:
        return ""

    rle = " ".join(
        f"{start} {length}"
        for start, length in zip(starts, lengths)
    )

    return rle

In [ ]:
# 16. Create inference DataLoader

inference_dataset = InferenceForgeryDataset(
    df=inference_df,
    target_size=IMAGE_SIZE
)

inference_loader = DataLoader(
    inference_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [ ]:
# 17. Run inference on all batches and build full submission dataframe

all_resized_predictions = []

model.eval()

with torch.no_grad():
    for batch_idx, inference_batch in enumerate(inference_loader):
        case_ids = inference_batch["case_id"]
        heights = inference_batch["original_size"][0]
        widths = inference_batch["original_size"][1]
        inference_images = inference_batch["image"].to(device)

        inference_logits = model(inference_images)
        inference_probs = torch.sigmoid(inference_logits)
        inference_pred_masks = (inference_probs >= THRESHOLD).float()

        for i in range(len(case_ids)):
            case_id = case_ids[i]

            pred_mask = inference_pred_masks[i].detach().cpu().squeeze(0).numpy()

            original_height = heights[i].item()
            original_width = widths[i].item()

            pred_mask_resized = cv2.resize(
                pred_mask,
                (original_width, original_height),
                interpolation=cv2.INTER_NEAREST
            )

            pred_mask_resized = (pred_mask_resized > 0).astype(np.uint8)

            all_resized_predictions.append(
                {
                    "case_id": case_id,
                    "original_size": (original_height, original_width),
                    "pred_mask_resized": pred_mask_resized,
                    "positive_pixels": int(pred_mask_resized.sum()),
                }
            )

        print(
            f"Processed batch {batch_idx + 1} / {len(inference_loader)} "
            f"| Total predictions so far: {len(all_resized_predictions)}"
        )

full_submission_rows = []

for pred in all_resized_predictions:
    case_id = pred["case_id"]
    pred_mask_resized = pred["pred_mask_resized"]

    rle = mask_to_rle(pred_mask_resized)

    full_submission_rows.append(
        {
            "id": case_id,
            "rle": rle if rle != "" else "authentic",
        }
    )

full_submission_df = pd.DataFrame(full_submission_rows)

print("\nFull submission dataframe shape:", full_submission_df.shape)

print("\nFirst 5 rows:")
display(full_submission_df.head())

print("\nSubmission summary:")
print("Rows marked as authentic:", (full_submission_df["rle"] == "authentic").sum())
print("Rows with encoded masks:", (full_submission_df["rle"] != "authentic").sum())

In [ ]:
# 18. Save full submission CSV

full_submission_path = OUTPUT_DIR / "full_submission.csv"

full_submission_df.to_csv(full_submission_path, index=False)

print("Full submission file saved to:")
print(full_submission_path)

print("\nSaved file exists:", full_submission_path.exists())

print("\nPreview of saved full submission:")
display(pd.read_csv(full_submission_path).head())